# The Negotiation Arena — Project Walkthrough & Findings

## What is this project about?
When we put two AI agents in a room to negotiate a deal, do they genuinely reason and adapt to the situation, or are they just blindly obeying the rules written in their system prompts?

To test this, we built a simulated marketplace where two **Gemini 3.6 Flash** agents (a Buyer and a Seller) negotiate the price of a used phone over up to 8 rounds.

### The 3 Test Scenarios
1. **Cooperative:** Both agents want a fair deal. The seller's minimum ($150) is lower than the buyer's maximum ($155). There is an overlap where both can say yes — this is the **ZOPA** (Zone of Possible Agreement).
2. **Deceptive:** The price boundaries are the same, but the seller is instructed to manipulate the buyer (fake urgency, guilt trips), while the buyer is told to call out bluffs.
3. **Competitive:** *Zero ZOPA*. The seller refuses to go below $170, but the buyer's maximum budget is $130. Agreement is mathematically impossible unless an agent breaks its rules.

In line with the course guidelines, the core engine lives in modular files (`agent.py`, `arena.py`, `scenario_config.py`, `analysis.py`) under `src/`. This notebook is our demonstration companion to load our 100 recorded runs, inspect what the agents said, compute the NLP metrics, and view the plots.

In [ ]:
import sys, os
import json
import glob
import re
import pandas as pd
import matplotlib.pyplot as plt

# Connect to our custom modules inside src/
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

from scenario_config import SCENARIO_BOUNDS
import analysis

print("Imports ready: All analysis tools loaded successfully.")

## 1. The Rules & Prompts: Setting Up the ZOPA

Before running a deal, we give each agent private price limits and behavioral instructions. Let's inspect the numbers to see where a deal is possible and where a deadlock is built-in.

In [ ]:
# Create an easy-to-read table of the scenario boundaries
overview = []
for name, b in SCENARIO_BOUNDS.items():
    s_min, s_max = b['seller']
    b_min, b_max = b['buyer']
    margin = b_max - s_min
    overview.append({
        "Scenario": name.capitalize(),
        "Seller Range": f"${s_min} - ${s_max}",
        "Buyer Range": f"${b_min} - ${b_max}",
        "ZOPA Margin": f"${margin}" if margin >= 0 else f"-${abs(margin)} (No Deal Zone)",
        "Deal Possible?": "Yes" if margin >= 0 else "No"
    })

pd.DataFrame(overview)

## 2. Inspecting a Real Negotiation Log

The arena forces both models to reply using a strict JSON format:
`{"price": <number>, "message": "<text>"}`

Let's load one of our saved pre-recorded runs from the `deceptive` scenario to see how the agents negotiate turn-by-turn.

In [ ]:
# Grab one recorded deceptive negotiation
files = glob.glob("../results/deceptive/*.json")
with open(files[0], "r", encoding="utf-8") as f:
    sample_deal = json.load(f)

# Display the turns cleanly
history = sample_deal.get('transcript', sample_deal.get('history', []))
turn_rows = []
last_offer = None

for i, t in enumerate(history, 1):
    speaker = t.get('speaker', f"Turn {i}")
    price = t.get('price')
    msg = t.get('message', '')
    concession = f"{price - last_offer:+d}$" if last_offer is not None else "Start"
    last_offer = price
    
    turn_rows.append({
        "Turn": i,
        "Speaker": speaker,
        "Offer": f"${price}",
        "Shift": concession,
        "Message": msg
    })

print(f"Outcome: {'AGREEMENT' if sample_deal.get('agreed') else 'IMPASSE'} at ${sample_deal.get('final_price')}\n")
pd.DataFrame(turn_rows)

## 3. How We Measure Language: The NLP Pipeline

To evaluate the dialogue quality as required by the course guidelines, we analyze the text across four key metrics:
1. **Flesch Reading Ease:** Readability score (higher means simpler, lower means more complex phrasing).
2. **Lexical Diversity (Type-Token Ratio):** Vocabulary richness (unique words / total words).
3. **VADER Sentiment:** Emotional tone score from -1.0 (very negative/hostile) to +1.0 (very positive).
4. **Rhetorical Tactics:** Automatic regex detection for pressure words like false urgency, guilt, threats, or callouts.

Let's test these functions live on two sample sentences from our data to see the difference.

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import textstat

vader = SentimentIntensityAnalyzer()

coop_phrase = "I appreciate your reasonable offer. Let's meet at $152 for a fair deal."
deceptive_phrase = "Another buyer is waiting right now. If you don't take $154 immediately, the deal is off."

def analyze_text(text):
    sentiment = vader.polarity_scores(text)['compound']
    flesch = textstat.flesch_reading_ease(text)
    has_urgency = bool(re.search(r'\b(now|hurry|waiting|another|today)\b', text, re.I))
    has_threat = bool(re.search(r'\b(walk away|deal is off|final offer)\b', text, re.I))
    
    return {
        "Text Example": text,
        "Sentiment (-1 to +1)": round(sentiment, 2),
        "Flesch Readability": round(flesch, 1),
        "Urgency Detected": has_urgency,
        "Threat Detected": has_threat
    }

pd.DataFrame([analyze_text(coop_phrase), analyze_text(deceptive_phrase)])

## 4. The Full Dataset (100 Negotiations)

Now we load the full experimental dataset (100 independent negotiation runs) to compare game outcomes and language patterns side-by-side:
* 20 Cooperative ($T = 0.7$)
* 20 Deceptive ($T = 0.7$)
* 20 Competitive Baseline ($T = 0.7$)
* 20 Competitive Cold ($T = 0.4$)
* 20 Competitive Creative ($T = 1.0$)

In [ ]:
# Caricamento di tutti i dati registrati
corpus_results = analysis.load_results('../results')
computed_metrics = {s: analysis.compute_metrics(runs, s) for s, runs in corpus_results.items()}

# Costruzione della tabella riassuntiva multi-metrica
summary_table = []
for scenario, m in computed_metrics.items():
    summary_table.append({
        "Condition": scenario,
        "Runs (N)": len(corpus_results[scenario]),
        "Agreement (%)": f"{m['agreement_rate'] * 100:.1f}%",
        "Avg Rounds": round(m['mean_rounds'], 2),
        "Mean Sentiment": round(m.get('sentiment_mean', 0), 2),
        "Flesch Score": round(m.get('flesch_reading_ease', 0), 1),
        "Coherence (TF-IDF)": round(m.get('tfidf_coherence', 0), 3),
        "Violations": m.get('constraint_violations', 0)
    })

df_summary = pd.DataFrame(summary_table).sort_values(by="Condition")
display(df_summary.reset_index(drop=True))

## 5. Visualizing the Report Charts

Let's generate and display the main plots created by `analysis.py`.

In [ ]:
# Generazione e salvataggio dei grafici analitici
analysis.plot_price_evolution(corpus_results, save_dir='../plots')
analysis.plot_linguistic_features(computed_metrics, save_dir='../plots')
analysis.plot_language_metrics(computed_metrics, save_dir='../plots')
analysis.plot_success_rates(computed_metrics, save_dir='../plots')

from IPython.display import Image, display

plots = ['success_rates.png', 'price_convergence.png', 'linguistic_features.png', 'language_metrics.png']
for p in plots:
    path = os.path.join('..', 'plots', p)
    if os.path.exists(path):
        print(f"--- Visualizing: {p} ---")
        display(Image(filename=path))

## 6. What Did We Find? 

1. **ZOPA:** Whenever a feasible price window exists (Cooperative and Deceptive), the agents always reach a compromise around $151. Even when the seller acts manipulative, the deal closes.
2. **Style vs. Strategy:** The model adopts the persona linguistically (negative tone, heavier sentences in Deceptive dialogues), but it does not let emotion alter the final agreed price.
3. **The Core Finding (Temperature Invariance):** When there is no ZOPA (Competitive), changing the sampling temperature from $0.4$ (rigid) to $1.0$ (creative) changes nothing. The failure rate remains at nearly 100%. 
   
This proves that the LLM is not weighing economic trade-offs flexibly; it treats the prompt's price limits as a **strict hard rule**, and prompt instructions completely override internal sampling parameters.